# ResNet-50 for Hand Sign Classification

This project implements a 50-layer Residual Network (ResNet-50) in TensorFlow/Keras for six-class hand sign image classification.

The implementation builds the residual architecture from reusable **identity blocks** and **convolutional blocks**, then trains the network on 64×64 RGB images. The notebook covers the full workflow: data preparation, model construction, training, evaluation, and single-image inference.

### Project highlights

- Implemented residual skip connections directly with the Keras Functional API.
- Built both identity and projection/convolutional residual blocks.
- Assembled the full ResNet-50 stage structure from those blocks.
- Trained and evaluated the model on a six-class hand sign dataset.
- Added a reusable inference helper for classifying individual images.


## Contents

1. Setup  
2. Dataset utilities  
3. Residual building blocks  
4. ResNet-50 architecture  
5. Data preparation  
6. Training  
7. Evaluation  
8. Single-image inference  
9. References


## 1. Setup

In [1]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from tensorflow.keras.initializers import glorot_uniform, random_uniform
from tensorflow.keras.layers import (
    Activation,
    Add,
    AveragePooling2D,
    BatchNormalization,
    Conv2D,
    Dense,
    Flatten,
    Input,
    MaxPooling2D,
    ZeroPadding2D,
)
from tensorflow.keras.models import Model

%matplotlib inline

np.random.seed(1)
tf.random.set_seed(1)


## 2. Dataset utilities

The dataset is stored in two HDF5 files: one for training and one for testing. The helper below loads the images, labels, and class identifiers without relying on external notebook utilities.


In [2]:
def load_signs_dataset(dataset_dir="datasets"):
    """Load the hand sign classification dataset from HDF5 files."""
    dataset_dir = Path(dataset_dir)
    train_path = dataset_dir / "train_signs.h5"
    test_path = dataset_dir / "test_signs.h5"

    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            "Expected 'train_signs.h5' and 'test_signs.h5' inside "
            f"{dataset_dir.resolve()}."
        )

    with h5py.File(train_path, "r") as train_file:
        x_train = np.array(train_file["train_set_x"][:])
        y_train = np.array(train_file["train_set_y"][:])

    with h5py.File(test_path, "r") as test_file:
        x_test = np.array(test_file["test_set_x"][:])
        y_test = np.array(test_file["test_set_y"][:])
        classes = np.array(test_file["list_classes"][:])

    return x_train, y_train, x_test, y_test, classes


def one_hot_encode(labels, num_classes):
    """Convert integer class labels to one-hot encoded vectors."""
    return tf.keras.utils.to_categorical(
        np.asarray(labels).reshape(-1),
        num_classes=num_classes,
    )


## 3. Residual building blocks

A residual block learns a transformation \(F(x)\) and combines it with a shortcut connection. The shortcut gives the network a direct path for information and gradients across many layers.

Two block types are used here:

- **Identity block:** the input and output dimensions already match, so the shortcut passes the input directly to the addition step.
- **Convolutional block:** the shortcut uses a \(1\times1\) convolution to match spatial or channel dimensions before addition.


In [3]:
def identity_block(X, f, filters, training=True):
    """Residual identity block with three convolutional layers."""
    F1, F2, F3 = filters
    X_shortcut = X

    # 1x1 convolution
    X = Conv2D(
        filters=F1,
        kernel_size=1,
        strides=(1, 1),
        padding="valid",
        kernel_initializer=random_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)
    X = Activation("relu")(X)

    # fxf convolution
    X = Conv2D(
        filters=F2,
        kernel_size=f,
        strides=(1, 1),
        padding="same",
        kernel_initializer=random_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)
    X = Activation("relu")(X)

    # 1x1 convolution
    X = Conv2D(
        filters=F3,
        kernel_size=1,
        strides=(1, 1),
        padding="valid",
        kernel_initializer=random_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)

    # Residual connection
    X = Add()([X_shortcut, X])
    X = Activation("relu")(X)

    return X


In [4]:
def convolutional_block(X, f, filters, s=2, training=True):
    """Residual block that projects the shortcut to a new shape."""
    F1, F2, F3 = filters
    X_shortcut = X

    # Main path
    X = Conv2D(
        filters=F1,
        kernel_size=1,
        strides=(s, s),
        padding="valid",
        kernel_initializer=glorot_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)
    X = Activation("relu")(X)

    X = Conv2D(
        filters=F2,
        kernel_size=f,
        strides=(1, 1),
        padding="same",
        kernel_initializer=glorot_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)
    X = Activation("relu")(X)

    X = Conv2D(
        filters=F3,
        kernel_size=1,
        strides=(1, 1),
        padding="valid",
        kernel_initializer=glorot_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X, training=training)

    # Projection shortcut
    X_shortcut = Conv2D(
        filters=F3,
        kernel_size=1,
        strides=(s, s),
        padding="valid",
        kernel_initializer=glorot_uniform(seed=0),
    )(X_shortcut)
    X_shortcut = BatchNormalization(axis=3)(X_shortcut, training=training)

    X = Add()([X, X_shortcut])
    X = Activation("relu")(X)

    return X


## 4. ResNet-50 architecture

The network starts with a \(7\times7\) convolution and max pooling, followed by four residual stages. Each stage begins with a convolutional block when a change in resolution or channel depth is required, then continues with identity blocks.

The stage layout used here is:

- Stage 2: 1 convolutional block + 2 identity blocks
- Stage 3: 1 convolutional block + 3 identity blocks
- Stage 4: 1 convolutional block + 5 identity blocks
- Stage 5: 1 convolutional block + 2 identity blocks

The final feature map is average-pooled, flattened, and passed to a softmax classifier.


In [5]:
def build_resnet50(input_shape=(64, 64, 3), classes=6):
    """Build a ResNet-50 image classifier using custom residual blocks."""
    X_input = Input(input_shape)

    # Stem
    X = ZeroPadding2D((3, 3))(X_input)
    X = Conv2D(
        64,
        (7, 7),
        strides=(2, 2),
        kernel_initializer=glorot_uniform(seed=0),
    )(X)
    X = BatchNormalization(axis=3)(X)
    X = Activation("relu")(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    # Stage 2
    X = convolutional_block(X, f=3, filters=[64, 64, 256], s=1)
    X = identity_block(X, 3, [64, 64, 256])
    X = identity_block(X, 3, [64, 64, 256])

    # Stage 3
    X = convolutional_block(X, f=3, filters=[128, 128, 512], s=2)
    X = identity_block(X, 3, [128, 128, 512])
    X = identity_block(X, 3, [128, 128, 512])
    X = identity_block(X, 3, [128, 128, 512])

    # Stage 4
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], s=2)
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])

    # Stage 5
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], s=2)
    X = identity_block(X, 3, [512, 512, 2048])
    X = identity_block(X, 3, [512, 512, 2048])

    # Classification head
    X = AveragePooling2D((2, 2))(X)
    X = Flatten()(X)
    X = Dense(
        classes,
        activation="softmax",
        kernel_initializer=glorot_uniform(seed=0),
    )(X)

    return Model(inputs=X_input, outputs=X, name="resnet50_sign_classifier")


In [6]:
model = build_resnet50(input_shape=(64, 64, 3), classes=6)
model.summary()


Model: "resnet50_sign_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 70, 70, 3) │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      9,472 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 15, 15,    │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 15, 15,    │      4,160 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 15, 15,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 15, 15,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 15, 15,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 15, 15,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 15, 15,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 15, 15,    │     16,640 │ activation_2[0][… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 15, 15,    │     16,640 │ max_pooling2d[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 15, 15,    │      1,024 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 15, 15,    │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 15, 15,    │          0 │ batch_normalizat

 Total params: 23,600,006 (90.03 MB)

 Trainable params: 23,546,886 (89.82 MB)

 Non-trainable params: 53,120 (207.50 KB)

## 5. Data preparation

In [8]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_signs_dataset()

X_train = X_train_orig.astype("float32") / 255.0
X_test = X_test_orig.astype("float32") / 255.0

Y_train = one_hot_encode(Y_train_orig, num_classes=6)
Y_test = one_hot_encode(Y_test_orig, num_classes=6)

print(f"Training examples: {X_train.shape[0]}")
print(f"Test examples: {X_test.shape[0]}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")


Training examples: 1080
Test examples: 120
X_train shape: (1080, 64, 64, 3)
Y_train shape: (1080, 6)
X_test shape: (120, 64, 64, 3)
Y_test shape: (120, 6)


## 6. Training

The model is optimized with Adam and categorical cross-entropy. Ten epochs provide a compact training run while still showing the network's ability to fit the dataset.


In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_train,
    Y_train,
    epochs=10,
    batch_size=32,
)


Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.4824 - loss: 2.0074
Epoch 2/10


## 7. Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")


4/4 [==============================] - 0s 8ms/step - loss: 0.0985 - accuracy: 0.9750
Test loss: 0.0985
Test accuracy: 97.5000%


### Result

In the recorded run above, the model reached **97.5% test accuracy** after 10 training epochs on the six-class hand sign dataset.


## 8. Single-image inference

In [ ]:
def predict_image(model, image_path, target_size=(64, 64)):
    """Load an image, run inference, and return the predicted class and probabilities."""
    img = tf.keras.utils.load_img(image_path, target_size=target_size)
    x = tf.keras.utils.img_to_array(img)
    x = np.expand_dims(x, axis=0) / 255.0

    probabilities = model.predict(x, verbose=0)[0]
    predicted_class = int(np.argmax(probabilities))

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted class: {predicted_class}")
    plt.show()

    return predicted_class, probabilities


# Example:
# predicted_class, probabilities = predict_image(model, "images/my_image.jpg")
# print("Predicted class:", predicted_class)
# print("Class probabilities:", probabilities)
